### Cleanup frequencies from cleanup logs

In [24]:
import platform

import pandas as pd

from lexical_benchmark import settings
from lexical_benchmark.datasets import childes, utils as dataset_utils
from lexical_benchmark.utils import timed_status

assert platform.node() in settings.PATH.KNOWN_HOSTS, "Code Running in a unknown device, you must provide custom PATH locations"

dataset = childes.CHILDESDataset()

raw_word_frequencies = {}
rejected_word_frequencies = {}
clean_word_frequencies = {}

for lang_accent in dataset.accents:
    for speech_type in dataset.speech_types:
        with timed_status(status=f"Fetching word_frequencies ({lang_accent}/{speech_type})...", complete_status=f"Succesfully fetched word frequencies for {lang_accent}/{speech_type} !!"):
            rejected_word_frequencies[f"{lang_accent}/{speech_type}"] = dataset.word_frequencies(
                    lang_accent=lang_accent, speech_type=speech_type, word_type="rejected"
            )
            raw_word_frequencies[f"{lang_accent}/{speech_type}"] = dataset.word_frequencies(
                    lang_accent=lang_accent, speech_type=speech_type, word_type="raw"
            )
            clean_word_frequencies[f"{lang_accent}/{speech_type}"] = dataset.word_frequencies(
                    lang_accent=lang_accent, speech_type=speech_type, word_type="clean"
            )

wr = []
for lang_accent in dataset.accents:
    for speech_type in dataset.speech_types:
        with timed_status(status=f"Computing Rates ({lang_accent}/{speech_type})...", complete_status=f"Succesfully computed rates for {lang_accent}/{speech_type} !!"):
            label = f"{lang_accent}/{speech_type}"

            # Load rejection frequencies 
            count = rejected_word_frequencies[f"{lang_accent}/{speech_type}"]
            rejection = pd.DataFrame.from_dict(count, orient="index").reset_index()
            rejection.columns = ["word", "freq"]

            # Load raw frequencies
            count = raw_word_frequencies[f"{lang_accent}/{speech_type}"]
            raw = pd.DataFrame.from_dict(count, orient="index").reset_index()
            raw.columns = ["word", "freq"]

            # Load clean frequencies
            count = clean_word_frequencies[f"{lang_accent}/{speech_type}"]
            clean = pd.DataFrame.from_dict(count, orient="index").reset_index()
            clean.columns = ["word", "freq"]

            wr.append({
                "Label": label,
                "Tokens": raw["freq"].sum(),
                "Tokens Rejected": rejection["freq"].sum(),
                "Token Rejection": rejection["freq"].sum() / raw["freq"].sum(),
                "Tokens Accepted": clean["freq"].sum(),
                "Token Acceptance": clean["freq"].sum() / raw["freq"].sum(),
                "Types": len(raw),
                "Types Rejected": len(rejection),
                "Type Rejection": len(rejection) / len(raw),
                "Types Accepted": len(clean),
                "Type Acceptance": len(clean) / len(raw),
            })


childes_global_word_rejection_rates = wr
%store childes_global_word_rejection_rates

Output()

Output()

Succesfully fetched word frequencies for Eng-NA/child !! (Total time: 16 seconds)

Output()

Succesfully fetched word frequencies for Eng-UK/adult !! (Total time: 13 seconds)

...

Output()

Succesfully computed rates for Eng-UK/adult !! (Total time: 1 seconds)

Output()

Succesfully computed rates for Eng-UK/child !! (Total time: 1 seconds)

Stored 'childes_global_word_rejection_rates' (list)

# Block Average Rejection Rates

To be able to compare those stats with STELA and other datasets of different size we do a block-avergage computation.

In [4]:
from lexical_benchmark.datasets.childes import data2 as childes
from lexical_benchmark.utils import timed_status

dataset = childes.CHILDESDataset()
childes_global_child_speech_words = {}
childes_global_adult_speech_words = {}

for lang_accent in dataset.accents:
    with timed_status(status=f"Gathering Words {lang_accent}", complete_status=f"Finished Gathering Words from {lang_accent}"):
        child_words = []
        adult_words = []
        for item in dataset.iter_accent(lang_accent):
            child_words.extend(item.preprocess_item("child").processed.read_tokenized())
            adult_words.extend(item.preprocess_item("adult").processed.read_tokenized())
        childes_global_child_speech_words[lang_accent] = child_words
        childes_global_adult_speech_words[lang_accent] = adult_words

%store childes_global_child_speech_words
%store childes_global_adult_speech_words

Output()

Finished Gathering Words from Eng-NA (Total time: 13 seconds)

Output()

Finished Gathering Words from Eng-UK (Total time: 8 seconds)

Stored 'childes_global_child_speech_words' (dict)
Stored 'childes_global_adult_speech_words' (dict)


### Word Rejection for full dictionnairy (all-extras)

Word Rejection Rate with the full dictionnairy of words.

In [1]:
%store -r childes_global_child_speech_words
%store -r childes_global_adult_speech_words
from lexical_benchmark.utils import timed_status
from lexical_benchmark.stats import block_average
from lexical_benchmark.datasets import utils

# Dictionairy of ALL WORDS (Child & Adult Extras)
en_cleaner = utils.DictionairyCleaner(lang="EN", childes_extra_id="9a30b8dad7abe369b2402b989f07e28b")

with timed_status(status="Computing Rates for Eng_NA/child", complete_status="Finished Eng_NA/child !"):
    words = childes_global_child_speech_words["Eng-NA"]
    child_na_rj_rate = block_average.calculate_block_word_filtering_rates(
        [words], dictionairy=en_cleaner
    )

with timed_status(status="Computing Rates for Eng_NA/adult", complete_status="Finished Eng_NA/adult !"):
    words = childes_global_adult_speech_words["Eng-NA"]
    adult_na_rj_rate = block_average.calculate_block_word_filtering_rates(
        [words], dictionairy=en_cleaner
    )

with timed_status(status="Computing Rates for Eng_UK/child", complete_status="Finished Eng_UK/child !"):
    words = childes_global_child_speech_words["Eng-UK"]
    child_uk_rj_rate = block_average.calculate_block_word_filtering_rates(
        [words], dictionairy=en_cleaner
    )
with timed_status(status="Computing Rates for Eng_UK/adult", complete_status="Finished Eng_UK/adult !"):
    words = childes_global_adult_speech_words["Eng-UK"]
    adult_uk_rj_rate = block_average.calculate_block_word_filtering_rates(
        [words], dictionairy=en_cleaner
    )

view_tokens_cfg = {"view_type": "result_tokens", "avg_type": "average"}
view_types_cfg = {"view_type": "result_types", "avg_type": "average"}
childes_word_stats_tables_allwords = {
    "tokens": {
        "Eng-NA/child": {"Section": "Eng-NA/child", **child_na_rj_rate.view(**view_tokens_cfg)},
        "Eng-NA/adult": {"Section": "Eng-NA/adult", **adult_na_rj_rate.view(**view_tokens_cfg)},
        "Eng-UK/child": {"Section": "Eng-UK/child", **child_uk_rj_rate.view(**view_tokens_cfg)},
        "Eng-UK/adult": {"Section": "Eng-UK/adult", **adult_uk_rj_rate.view(**view_tokens_cfg)},
    },
    "types": {
        "Eng-NA/child": {"Section": "Eng-NA/child", **child_na_rj_rate.view(**view_types_cfg)},
        "Eng-NA/adult": {"Section": "Eng-NA/adult", **adult_na_rj_rate.view(**view_types_cfg)},
        "Eng-UK/child": {"Section": "Eng-UK/child", **child_uk_rj_rate.view(**view_types_cfg)},
        "Eng-UK/adult": {"Section": "Eng-UK/adult", **adult_uk_rj_rate.view(**view_types_cfg)},
    }
}
%store childes_word_stats_tables_allwords

Output()

Finished Eng_NA/child ! (Total time: 3 seconds)

Output()

Finished Eng_NA/adult ! (Total time: 6 seconds)

Output()

Finished Eng_UK/child ! (Total time: 2 seconds)

Output()

Finished Eng_UK/adult ! (Total time: 6 seconds)

Stored 'childes_word_stats_tables_allwords' (dict)


#### Pass all words through dictionairy and keep reject/acceptance rates

In [2]:
%store -r childes_global_child_speech_words
%store -r childes_global_adult_speech_words
from lexical_benchmark.utils import timed_status
from lexical_benchmark.stats import block_average
from lexical_benchmark.datasets import utils

CHUNK_SIZE = 16_000
en_cleaner = utils.DictionairyCleaner(lang="EN", childes_extra_id="36c92a5e6bc76949cfe5a0336c1152df")

with timed_status(status="Computing Rates for Eng_NA/child", complete_status="Finished Eng_NA/child !"):
    words = childes_global_child_speech_words["Eng-NA"]
    word_chunk_list = block_average.split_and_fill_chunks(words, chunk_size=CHUNK_SIZE)
    child_na_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )

with timed_status(status="Computing Rates for Eng_NA/adult", complete_status="Finished Eng_NA/adult !"):
    words = childes_global_adult_speech_words["Eng-NA"]
    word_chunk_list = block_average.split_and_fill_chunks(words, chunk_size=CHUNK_SIZE)
    adult_na_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )

with timed_status(status="Computing Rates for Eng_UK/child", complete_status="Finished Eng_UK/child !"):
    words = childes_global_child_speech_words["Eng-UK"]
    word_chunk_list = block_average.split_and_fill_chunks(words, chunk_size=CHUNK_SIZE)
    child_uk_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )
with timed_status(status="Computing Rates for Eng_UK/adult", complete_status="Finished Eng_UK/adult !"):
    words = childes_global_adult_speech_words["Eng-UK"]
    word_chunk_list = block_average.split_and_fill_chunks(words, chunk_size=CHUNK_SIZE)
    adult_uk_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )

view_cfg = {"view_type": "result_tokens", "avg_type": "average"}
childes_word_stats_tables_16k_tokens = {
    "Eng-NA/child": {"Section": "Eng-NA/child", **child_na_rj_rate.view(**view_cfg)},
    "Eng-NA/adult": {"Section": "Eng-NA/adult", **adult_na_rj_rate.view(**view_cfg)},
    "Eng-UK/child": {"Section": "Eng-UK/child", **child_uk_rj_rate.view(**view_cfg)},
    "Eng-UK/adult": {"Section": "Eng-UK/adult", **adult_uk_rj_rate.view(**view_cfg)},
}
view_cfg = {"view_type": "result_types", "avg_type": "average"}
childes_word_stats_tables_16k_types = {
    "Eng-NA/child": {"Section": "Eng-NA/child", **child_na_rj_rate.view(**view_cfg)},
    "Eng-NA/adult": {"Section": "Eng-NA/adult", **adult_na_rj_rate.view(**view_cfg)},
    "Eng-UK/child": {"Section": "Eng-UK/child", **child_uk_rj_rate.view(**view_cfg)},
    "Eng-UK/adult": {"Section": "Eng-UK/adult", **adult_uk_rj_rate.view(**view_cfg)},
}
%store childes_word_stats_tables_16k_tokens
%store childes_word_stats_tables_16k_types

Output()

Finished Eng_NA/child ! (Total time: 3 seconds)

Output()

Finished Eng_NA/adult ! (Total time: 6 seconds)

Output()

Finished Eng_UK/child ! (Total time: 2 seconds)

Output()

Finished Eng_UK/adult ! (Total time: 6 seconds)

Stored 'childes_word_stats_tables_16k_tokens' (dict)
Stored 'childes_word_stats_tables_16k_types' (dict)


In [3]:
%store -r childes_global_child_speech_words
%store -r childes_global_adult_speech_words
from lexical_benchmark.utils import timed_status
from lexical_benchmark.stats import block_average
from lexical_benchmark.datasets import utils

CHUNK_SIZE = 1600
en_cleaner = utils.DictionairyCleaner(lang="EN", childes_extra_id="36c92a5e6bc76949cfe5a0336c1152df")

with timed_status(status="Computing Rates for Eng_NA/child", complete_status="Finished Eng_NA/child !"):
    words = childes_global_child_speech_words["Eng-NA"]
    word_chunk_list = block_average.split_and_fill_chunks(words, chunk_size=CHUNK_SIZE)
    child_na_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )

with timed_status(status="Computing Rates for Eng_NA/adult", complete_status="Finished Eng_NA/adult !"):
    words = childes_global_adult_speech_words["Eng-NA"]
    word_chunk_list = block_average.split_and_fill_chunks(words, chunk_size=CHUNK_SIZE)
    adult_na_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )

with timed_status(status="Computing Rates for Eng_UK/child", complete_status="Finished Eng_UK/child !"):
    words = childes_global_child_speech_words["Eng-UK"]
    word_chunk_list = block_average.split_and_fill_chunks(words, chunk_size=CHUNK_SIZE)
    child_uk_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )
with timed_status(status="Computing Rates for Eng_UK/adult", complete_status="Finished Eng_UK/adult !"):
    words = childes_global_adult_speech_words["Eng-UK"]
    word_chunk_list = block_average.split_and_fill_chunks(words, chunk_size=CHUNK_SIZE)
    adult_uk_rj_rate = block_average.calculate_block_word_filtering_rates(
        word_chunk_list, dictionairy=en_cleaner
    )

view_cfg = {"view_type": "result_tokens", "avg_type": "average"}
childes_word_stats_tables_1k6h_tokens = {
    "Eng-NA/child": {"Section": "Eng-NA/child", **child_na_rj_rate.view(**view_cfg)},
    "Eng-NA/adult": {"Section": "Eng-NA/adult", **adult_na_rj_rate.view(**view_cfg)},
    "Eng-UK/child": {"Section": "Eng-UK/child", **child_uk_rj_rate.view(**view_cfg)},
    "Eng-UK/adult": {"Section": "Eng-UK/adult", **adult_uk_rj_rate.view(**view_cfg)},
}
view_cfg = {"view_type": "result_types", "avg_type": "average"}
childes_word_stats_tables_1k6h_types = {
    "Eng-NA/child": {"Section": "Eng-NA/child", **child_na_rj_rate.view(**view_cfg)},
    "Eng-NA/adult": {"Section": "Eng-NA/adult", **adult_na_rj_rate.view(**view_cfg)},
    "Eng-UK/child": {"Section": "Eng-UK/child", **child_uk_rj_rate.view(**view_cfg)},
    "Eng-UK/adult": {"Section": "Eng-UK/adult", **adult_uk_rj_rate.view(**view_cfg)},
}
%store childes_word_stats_tables_1k6h_tokens
%store childes_word_stats_tables_1k6h_types

Output()

Finished Eng_NA/child ! (Total time: 3 seconds)

Output()

Finished Eng_NA/adult ! (Total time: 6 seconds)

Output()

Finished Eng_UK/child ! (Total time: 3 seconds)

Output()

Finished Eng_UK/adult ! (Total time: 7 seconds)

Stored 'childes_word_stats_tables_1k6h_tokens' (dict)
Stored 'childes_word_stats_tables_1k6h_types' (dict)


In [6]:
%store -r childes_word_stats_tables_1k6h_tokens
%store -r childes_word_stats_tables_1k6h_types
%store -r childes_word_stats_tables_16k_tokens
%store -r childes_word_stats_tables_16k_types
%store -r childes_global_word_rejection_rates
%store -r childes_word_stats_tables_allwords
from IPython.display import display, HTML, display_html
import pandas as pd

from lexical_benchmark.utils import ipython_utils

display_html(HTML("<h3> CHILDES Rejection Rates ALL WORDS </h3><p> Using dictionairy with extras from adult & child.</p>"))
# ALL Words Stats
ipython_utils.display_side_by_side(
    dataframes={
        "": (
            ("Tokens", pd.DataFrame(list(childes_word_stats_tables_allwords["tokens"].values()))),
            ("Types", pd.DataFrame(list(childes_word_stats_tables_allwords["types"].values())))
        ),
    },
    custom_format={
        'Tokens': '{:,}',
        'Tokens Rejected': '{:,}',
        'Token Rejection': '{:.2%}',
        'Tokens Accepted': '{:,}',
        'Token Acceptance': '{:.2%}',
        'Types': '{:,}',
        'Types Rejected': '{:,}',
        'Type Rejection': '{:.2%}',
        'Types Accepted': '{:,}',
        'Type Acceptance': '{:.2%}'
    },
)

df = pd.DataFrame(childes_global_word_rejection_rates)
df = df.iloc[[1, 0, 3, 2]]
st_df = df.style.format({
    "Tokens": "{:,}",
    "Token Rejection": "{:.2%}",
    "Tokens Rejected": "{:,}",
    "Token Rejection": "{:.2%}",
    "Tokens Accepted": "{:,}",
    "Token Acceptance": "{:.2%}",
    "Types": "{:,}",
    "Types Rejected": "{:,}",
    "Type Rejection": "{:.2%}",
    "Types Accepted": "{:,}",
    "Type Acceptance": "{:.2%}",
})

display_html(HTML("<h3> Simple Rejection Rates </h3>"))
display(st_df)

ipython_utils.display_side_by_side(
    dataframes={
        "CHILDES 16k Block Average": (
            ("Tokens", pd.DataFrame(list(childes_word_stats_tables_16k_tokens.values()))),
            ("Types", pd.DataFrame(list(childes_word_stats_tables_16k_types.values())))
        ),
        "CHILDES 1.6k Block Average": (
            ("Tokens", pd.DataFrame(list(childes_word_stats_tables_1k6h_tokens.values()))),
            ("Types", pd.DataFrame(list(childes_word_stats_tables_1k6h_types.values())))
        ),
    },
    custom_format={
        'Tokens': '{:,}',
        'Tokens Rejected': '{:,}',
        'Token Rejection': '{:.2%}',
        'Tokens Accepted': '{:,}',
        'Token Acceptance': '{:.2%}',
        'Types': '{:,}',
        'Types Rejected': '{:,}',
        'Type Rejection': '{:.2%}',
        'Types Accepted': '{:,}',
        'Type Acceptance': '{:.2%}'
    },
)

CHILDES Rejection Rates ALL WORDS Using dictionairy with extras from adult & child.

,Section,Tokens,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance
0,Eng-NA/child,"2,908,126","30,687",1.06%,"2,877,439",98.94%
1,Eng-NA/adult,"7,953,776","36,912",0.46%,"7,916,864",99.54%
2,Eng-UK/child,"2,965,974","11,497",0.39%,"2,954,477",99.61%
3,Eng-UK/adult,"8,041,204","20,582",0.26%,"8,020,622",99.74%
,Section,Types,Types Rejected,Type Rejection,Types Accepted,Type Acceptance
0,Eng-NA/child,"35,502","8,549",24.08%,"26,953",75.92%
1,Eng-NA/adult,"41,442","7,590",18.31%,"33,852",81.69%
2,Eng-UK/child,"22,757","3,173",13.94%,"19,584",86.06%
3,Eng-UK/adult,"31,269","4,184",13.38%,"27,085",86.62%


Simple Rejection Rates

,Label,Tokens,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance,Types,Types Rejected,Type Rejection,Types Accepted,Type Acceptance
1,Eng-NA/child,"2,908,126","39,299",1.35%,"2,868,827",98.65%,"35,502","13,768",38.78%,"21,734",61.22%
0,Eng-NA/adult,"7,953,776","38,293",0.48%,"7,915,483",99.52%,"41,442","7,755",18.71%,"33,687",81.29%
3,Eng-UK/child,"2,965,974","18,427",0.62%,"2,947,547",99.38%,"22,757","5,928",26.05%,"16,829",73.95%
2,Eng-UK/adult,"8,041,204","23,252",0.29%,"8,017,952",99.71%,"31,269","4,292",13.73%,"26,977",86.27%


,Section,Tokens,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance
0,Eng-NA/child,"2,896,000","39,072",1.35%,"2,856,928",98.65%
1,Eng-NA/adult,"7,952,000","38,282",0.48%,"7,913,718",99.52%
2,Eng-UK/child,"2,960,000","18,395",0.62%,"2,941,605",99.38%
3,Eng-UK/adult,"8,032,000","23,224",0.29%,"8,008,776",99.71%
,Section,Types,Types Rejected,Type Rejection,Types Accepted,Type Acceptance
0,Eng-NA/child,"281,272","19,813",6.47%,"261,459",93.53%
1,Eng-NA/adult,"724,462","15,615",2.15%,"708,847",97.85%
2,Eng-UK/child,"257,394","8,705",3.03%,"248,689",96.97%
3,Eng-UK/adult,"734,257","9,963",1.34%,"724,294",98.66%
,Section,Tokens,Tokens Rejected,Token Rejection,Tokens Accepted,Token Acceptance
